# Task 3: Store Cleaned Data in PostgreSQL

**Objective:** Load the processed bank review data into a relational PostgreSQL database
with two tables — `banks` (entity master) and `reviews` (fact table).

**Schema:**
```
banks   → bank_id (PK), bank_name, app_name
reviews → review_id (PK), bank_id (FK), review_text, rating, review_date,
           sentiment_label, sentiment_score, identified_theme, source
```

In [ ]:
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv
from psycopg2.extras import execute_values

os.chdir('/home/code0053/fintech-review-analytics')
load_dotenv()   # reads credentials from .env (never committed to git)

DB_CONFIG = {
    "host":     os.getenv("DB_HOST", "localhost"),
    "port":     int(os.getenv("DB_PORT", 5433)),
    "database": os.getenv("DB_NAME", "bank_reviews"),
    "user":     os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
}
print("Config loaded. Connecting to:", DB_CONFIG["database"], "on port", DB_CONFIG["port"])

In [ ]:
try:
    conn = psycopg2.connect(**DB_CONFIG)
    print("Connected to PostgreSQL successfully!")
    conn.close()
except Exception as e:
    print("Connection failed:", e)

## Step 1 — Load the Cleaned Data

Read `bank_reviews_sentiment.csv` and split it into two DataFrames that match
the target table structure — exactly like the `restaurants.csv` / `reviews.csv`
pattern from the instruction.

In [ ]:
# Load the processed review data
df = pd.read_csv('data/raw/bank_reviews_sentiment.csv')
df['review_date'] = pd.to_datetime(df['date'], errors='coerce').dt.date

print(f"Total rows loaded: {len(df)}")
print(df.head(3).to_string())

# ── banks DataFrame  (matches restaurants.csv pattern) ──────
APP_IDS = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia":           "com.boa.boaMobileBanking",
    "Dashen Bank":                 "com.dashen.dashensuperapp",
}
banks_df = pd.DataFrame([
    {"bank_name": name, "app_name": app_id}
    for name, app_id in APP_IDS.items()
])
print("\nbanks_df:")
print(banks_df.to_string(index=False))

# ── reviews DataFrame (matches reviews.csv pattern) ─────────
reviews_df = df[['review', 'rating', 'review_date', 'bank',
                  'sentiment_label', 'sentiment_score',
                  'identified_theme', 'source']].copy()
reviews_df = reviews_df.rename(columns={'review': 'review_text'})
print(f"\nreviews_df shape: {reviews_df.shape}")
print(reviews_df.head(3).to_string(index=False))

## Step 2 — Create Tables

In [ ]:
CREATE_BANKS_TABLE = """
CREATE TABLE IF NOT EXISTS banks (
    bank_id   SERIAL PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL UNIQUE,
    app_name  VARCHAR(200)
);
"""

CREATE_REVIEWS_TABLE = """
CREATE TABLE IF NOT EXISTS reviews (
    review_id        SERIAL PRIMARY KEY,
    bank_id          INTEGER REFERENCES banks(bank_id) ON DELETE CASCADE,
    review_text      TEXT,
    rating           SMALLINT CHECK (rating BETWEEN 1 AND 5),
    review_date      DATE,
    sentiment_label  VARCHAR(20),
    sentiment_score  NUMERIC(6,4),
    identified_theme VARCHAR(100),
    source           VARCHAR(50)
);
"""

conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()
cur.execute(CREATE_BANKS_TABLE)
cur.execute(CREATE_REVIEWS_TABLE)
conn.commit()
cur.close()
conn.close()
print("Tables created successfully.")

## Step 3 — Insert Data

### 3a. Insert Banks

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

for _, row in banks_df.iterrows():
    cur.execute(
        "INSERT INTO banks (bank_name, app_name) VALUES (%s, %s) ON CONFLICT (bank_name) DO NOTHING;",
        (row['bank_name'], row['app_name'])
    )
conn.commit()

cur.execute("SELECT * FROM banks;")
print("banks table:")
for row in cur.fetchall():
    print(" ", row)

cur.close()
conn.close()

### 3b. Insert Reviews

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# Fetch bank_id lookup
cur.execute("SELECT bank_name, bank_id FROM banks;")
bank_id_map = dict(cur.fetchall())

# Build records list
records = []
for _, row in reviews_df.iterrows():
    bank_id = bank_id_map.get(row['bank'])
    if bank_id is None:
        continue
    records.append((
        bank_id,
        str(row['review_text'])[:2000] if pd.notna(row['review_text']) else None,
        int(row['rating']),
        row['review_date'],
        row['sentiment_label'],
        float(row['sentiment_score']),
        row['identified_theme'],
        row['source'],
    ))

INSERT_SQL = """
    INSERT INTO reviews
        (bank_id, review_text, rating, review_date,
         sentiment_label, sentiment_score, identified_theme, source)
    VALUES %s
    ON CONFLICT DO NOTHING;
"""

execute_values(cur, INSERT_SQL, records)
conn.commit()
print(f"Inserted {len(records)} reviews.")
cur.close()
conn.close()

## Step 4 — Verify Data Integrity

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

print("=== Count reviews per bank ===")
cur.execute("""
    SELECT b.bank_name, COUNT(r.review_id) AS total_reviews
    FROM banks b
    LEFT JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name
    ORDER BY total_reviews DESC;
""")
print(pd.DataFrame(cur.fetchall(), columns=['bank_name','total_reviews']).to_string(index=False))

print("\n=== Average rating per bank ===")
cur.execute("""
    SELECT b.bank_name, ROUND(AVG(r.rating)::numeric, 2) AS avg_rating
    FROM banks b
    JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name
    ORDER BY avg_rating DESC;
""")
print(pd.DataFrame(cur.fetchall(), columns=['bank_name','avg_rating']).to_string(index=False))

print("\n=== Null check on key columns ===")
cur.execute("""
    SELECT
        COUNT(*) FILTER (WHERE review_text IS NULL)     AS null_text,
        COUNT(*) FILTER (WHERE rating IS NULL)          AS null_rating,
        COUNT(*) FILTER (WHERE sentiment_label IS NULL) AS null_sentiment
    FROM reviews;
""")
print(pd.DataFrame(cur.fetchall(), columns=['null_text','null_rating','null_sentiment']).to_string(index=False))

cur.close()
conn.close()